In [1]:

from sentence_transformers import SentenceTransformer
import numpy as np
import pickle

In [2]:
#Load the cleaned text chunks
with open('financial_text_chunks.txt', 'r') as f:
    text_chunks = [line.strip() for line in f.readlines()]
print(f"Loaded {len(text_chunks)} text chunks.")

Loaded 619023 text chunks.


In [3]:
print(text_chunks[:5])  # Display first 5 chunks for verification

['On 2013-02-19, the stock AAL opened at $14.33, closed at $14.26, had a high of $14.56, a low of $14.08, and traded a volume of 11,354,400.', 'On 2013-02-20, the stock AAL opened at $14.17, closed at $13.33, had a high of $14.26, a low of $13.15, and traded a volume of 14,725,200.', 'On 2013-02-21, the stock AAL opened at $13.62, closed at $13.37, had a high of $13.95, a low of $12.90, and traded a volume of 11,922,100.', 'On 2013-02-22, the stock AAL opened at $13.57, closed at $13.57, had a high of $13.60, a low of $13.21, and traded a volume of 6,071,400.', 'On 2013-02-25, the stock AAL opened at $13.60, closed at $13.02, had a high of $13.76, a low of $13.00, and traded a volume of 7,186,400.']


In [4]:
#Load the SentenceTransformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
#Generate embeddings for the text chunks
embeddings = model.encode(text_chunks, show_progress_bar=False)
#Save the embeddings to a file

/home/abdul/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [5]:
output = {    'text_chunks': text_chunks,
    'embeddings': embeddings
}

In [7]:
with open('financial_embeddings.pkl', 'wb') as f:
    pickle.dump(output, f)
print(f"Generated embeddings for {len(text_chunks)} text chunks and saved to 'financial_embeddings.pkl'.")


Generated embeddings for 619023 text chunks and saved to 'financial_embeddings.pkl'.


In [8]:
print(text_chunks[:5])  # Display first 5 chunks for verification

['On 2013-02-19, the stock AAL opened at $14.33, closed at $14.26, had a high of $14.56, a low of $14.08, and traded a volume of 11,354,400.', 'On 2013-02-20, the stock AAL opened at $14.17, closed at $13.33, had a high of $14.26, a low of $13.15, and traded a volume of 14,725,200.', 'On 2013-02-21, the stock AAL opened at $13.62, closed at $13.37, had a high of $13.95, a low of $12.90, and traded a volume of 11,922,100.', 'On 2013-02-22, the stock AAL opened at $13.57, closed at $13.57, had a high of $13.60, a low of $13.21, and traded a volume of 6,071,400.', 'On 2013-02-25, the stock AAL opened at $13.60, closed at $13.02, had a high of $13.76, a low of $13.00, and traded a volume of 7,186,400.']


In [12]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# User input
query = input("Enter a phrase to check similarity: ")
query_emb = model.encode([query])[0]

# Compute similarities
sims = [cosine_similarity(query_emb, emb) for emb in embeddings]
max_idx = np.argmax(sims)
max_sim = sims[max_idx]

# Threshold for "I don't know"
threshold = 0.6

if max_sim > threshold:
    print(f"Most similar chunk (score={max_sim:.2f}):\n{text_chunks[max_idx]}")
else:
    print("I don't know.")

/home/abdul/.local/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


I don't know.


In [14]:
import pickle

# Load the file you saved earlier
with open("financial_embeddings.pkl", "rb") as f:
    data = pickle.load(f)

text_chunks = data["text_chunks"]
embeddings = data["embeddings"]


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

# Pick any two indices to compare
idx_1 = 0  # First sentence
idx_2 = 10  # Maybe similar sentence
idx_3 = 5000  # Distant topic sentence

# Compute cosine similarities
sim_1_2 = cosine_similarity([embeddings[idx_1]], [embeddings[idx_2]])[0][0]
sim_1_3 = cosine_similarity([embeddings[idx_1]], [embeddings[idx_3]])[0][0]

print("📌 Sentence 1:", text_chunks[idx_1])
print("📌 Sentence 2:", text_chunks[idx_2])
print("Cosine Similarity (1 vs 2):", sim_1_2)

print("\n📌 Sentence 3:", text_chunks[idx_3])
print("Cosine Similarity (1 vs 3):", sim_1_3)


📌 Sentence 1: On 2013-02-19, the stock AAL opened at $14.33, closed at $14.26, had a high of $14.56, a low of $14.08, and traded a volume of 11,354,400.
📌 Sentence 2: On 2013-03-05, the stock AAL opened at $14.01, closed at $14.05, had a high of $14.05, a low of $13.71, and traded a volume of 7,676,100.
Cosine Similarity (1 vs 2): 0.9719932

📌 Sentence 3: On 2017-12-26, the stock ABBV opened at $98.15, closed at $97.75, had a high of $98.36, a low of $97.05, and traded a volume of 2,364,528.
Cosine Similarity (1 vs 3): 0.72341216
